In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import colors
from matplotlib import pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from scipy import sparse
from scipy.interpolate import interp1d
from scipy.sparse.linalg import spsolve
from lmfit.models import LinearModel, LorentzianModel
from joblib import Parallel, delayed
import dataclasses

formatter = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
logging.basicConfig(level=logging.INFO, format=formatter)
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")

In [2]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
G_PEAK_RANGE = (1550, 1750)
D_PEAK_RANGE = (1300, 1400)
TWO_D_PEAK_RANGE = (2550, 2800)

In [3]:
@dataclasses.dataclass
class peaksetup:
    center: float
    weight: float
    label: str
    min: float | None = None
    max: float | None = None

In [4]:
def load_mapping_ascii(
    filepath: Path,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    with open(filepath, "r") as f:
        lines = f.readlines()

    header = lines[0].strip().split()
    wavenumber = np.array([float(x) for x in header])

    data = []
    for line in lines[1:]:
        parts = line.strip().split()
        if len(parts) == 0:
            continue

        try:
            row = [float(x) for x in parts]
            data.append(row)
        except ValueError:
            logger.warning(f"Skipping malformed line: {line.strip()}")
            continue

    data = np.array(data)
    x = data[:, 0]
    y = data[:, 1]
    spectra = data[:, 2:]

    return x, y, wavenumber, spectra

In [5]:
def correct_wavenumber_shift(
    spectra,
    wavenumber,
    si_range=(500, 600),
    target=520.7,
):
    spectra = np.atleast_2d(spectra)
    N_pixel, N_wn = spectra.shape

    # --- Siピーク範囲 ---
    mask = (wavenumber >= si_range[0]) & (wavenumber <= si_range[1])
    wn_si = wavenumber[mask]

    corrected = np.zeros_like(spectra)
    shifts = np.zeros(N_pixel)

    for i in range(N_pixel):
        spec = spectra[i, mask]

        # --- ピーク位置 ---
        peak_idx = np.argmax(spec)
        peak_pos = wn_si[peak_idx]

        # --- シフト量 ---
        shift = peak_pos - target
        shifts[i] = shift

        # --- 補間関数 ---
        f = interp1d(
            wavenumber - shift,
            spectra[i],
            kind="linear",
            bounds_error=False,
            fill_value="extrapolate",  # type: ignore
        )

        corrected[i] = f(wavenumber)

    return corrected, shifts

In [6]:
def fit_single_spectrum(x, y, peaks_setup: list[peaksetup]):
    """1ピクセル分のフィッティングを実行するヘルパー関数"""
    model = LinearModel(prefix='bkg_')
    params = model.make_params(slope=0, intercept=np.min(y))

    for i, p_config in enumerate(peaks_setup):
        prefix = f'p{i}_'
        p_model = LorentzianModel(prefix=prefix)
        model += p_model
        params.update(p_model.make_params())
        
        
        # 中心値の制約
        c_val = p_config.center
        params[f'{prefix}center'].set(
            value=c_val,
            min=p_config.min if p_config.min is not None else c_val - 20,
            max=p_config.max if p_config.max is not None else c_val + 20
        )
        
        idx = (np.abs(x - p_config.center)).argmin()
        actual_height = y[idx] - np.min(y)
        initial_sigma = 15
        initial_amplitude = actual_height * np.pi * initial_sigma
        params[f'{prefix}amplitude'].set(
            value=initial_amplitude, 
            min=0

        )
        params[f'{prefix}sigma'].set(value=initial_sigma, min=3, max=100)

    return model.fit(y, params, x=x)

In [7]:
def baseline_als(y, lam=1e5, p=0.01, niter=10):
    if niter <= 0:
        return y.copy()

    L = len(y)
    D = sparse.diags(
        [1.0, -2.0, 1.0],
        [0, -1, -2],  # type: ignore
        shape=(L, L - 2),
        dtype=float,
    )
    W = sparse.eye(L, dtype=float)
    z = y.copy()

    for _ in range(niter):
        W.setdiag(p * (y > 0) + (1 - p) * (y < 0))
        Z = (W + lam * D @ D.T).tocsc()  # ←重要
        z = spsolve(Z, W @ y)
    return z

def preprocess_baseline(spectra, wavenumber=None, target_range=(800, 3000)):
    corrected = spectra.copy()

    if wavenumber is not None and target_range is not None:
        mask = (wavenumber >= target_range[0]) & (wavenumber <= target_range[1])
    else:
        mask = np.ones(spectra.shape[1], dtype=bool)

    for i in range(len(spectra)):
        spec_cropped = spectra[i, mask]
        baseline = baseline_als(spec_cropped)
        corrected[i, mask] = spec_cropped - baseline
        corrected[i, ~mask] = 0
        
    return corrected

In [8]:
def create_ig_mask(IG, threshold=None, method="absolute", ratio=0.1):
    if method == "absolute":
        if threshold is None:
            raise ValueError("threshold must be specified for absolute mode")
        mask = IG > threshold

    elif method == "relative":
        thr = IG.max() * ratio
        mask = IG > thr

    else:
        raise ValueError(f"Unknown method: {method}")

    return mask

In [9]:
def apply_mask(data, mask):
    data = data.copy()
    data[~mask] = np.nan
    return data

In [10]:
def save_ratio_map_figure(
    filepath: Path,
    x,
    y,
    ratio,
    title="2D/G Ratio Map",
    cmap="viridis",
    vmin=0.0,
    vmax=2.0,
):
    filepath.parent.mkdir(parents=True, exist_ok=True)
    x = np.unique(x)
    y = np.unique(y)

    try:
        Z = ratio.reshape(len(y), len(x))
    except ValueError:
        logger.warning("Data size mismatch. Checking data...")
        Z = ratio

    Z_masked = np.ma.masked_invalid(Z)
    ratio = np.ma.masked_invalid(Z_masked)

    # Use plt.get_cmap to avoid MatplotlibDeprecationWarning
    cmap_obj = plt.get_cmap(cmap)
    try:
        cmap_obj.set_bad("k")
    except Exception:
        try:
            cmap_obj = colors.ListedColormap(cmap_obj(np.linspace(0, 1, 256)))
            cmap_obj.set_bad("k")
        except Exception:
            # Fallback: use 'viridis' ListedColormap with bad set to black
            fallback = plt.get_cmap("viridis")
            cmap_obj = colors.ListedColormap(fallback(np.linspace(0, 1, 256)))
            cmap_obj.set_bad("k")

    mesh = plt.pcolormesh(
        x, y, ratio, cmap=cmap_obj, vmin=vmin, vmax=vmax, shading="auto"
    )
    plt.colorbar(mesh, label=title)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.title(title)
    plt.savefig(filepath)
    plt.close()

In [11]:
def spectra_draw(
    wavenumber,
    spectra,
    fit_results_dict=None,
    spectra_ref=None,
    n=5,
    idx=None,
    mask_range=(0, 3000), # 描画範囲をグラフェンの主要ピークに合わせて調整
    g_range=None,
    d_range=None,
    two_d_range=None,
    show_peak=False,
    output_dir=Path("output/spectra_mapping/debug"),
    output_prefix="spectra_debug",
):
    output_dir.mkdir(parents=True, exist_ok=True)
    spectra = np.atleast_2d(spectra)
    
    # 描画するインデックスの決定
    if idx is None:
        available_idx = list(fit_results_dict.keys()) if fit_results_dict else range(len(spectra))
        idx = np.random.choice(available_idx, min(n, len(available_idx)), replace=False)

    for i in idx:
        plt.figure(figsize=(10, 6))
        
        # マスク範囲のデータ抽出（生データ表示用）
        mask = (wavenumber >= mask_range[0]) & (wavenumber <= mask_range[1])
        wn_plot = wavenumber[mask]
        spec_plot = spectra[i, mask]

        # --- 1. 生データのプロット ---
        plt.plot(wn_plot, spec_plot, color="blue", alpha=0.3, label="Raw Data", zorder=1)

        # --- 2. 比較用データのプロット ---
        if spectra_ref is not None:
            spec_ref_plot = spectra_ref[i, mask]
            plt.plot(wn_plot, spec_ref_plot, color="gray", alpha=0.4, label="Reference")
                        
        # --- 3. フィッティング結果の描画 ---
        if fit_results_dict is not None and i in fit_results_dict:
            res_list = fit_results_dict[i]
            
            for key, res in res_list.items():
                x_fit = res.userkws['x']
                y_fit = res.best_fit
                
                plt.plot(x_fit, y_fit, label=f"Best Fit ({key.upper()})", linewidth=2, zorder=3)
                comps = res.eval_components()
                bkg_val = comps.get('bkg_', np.zeros_like(x_fit))
                
                # 背景曲線の描画
                plt.plot(x_fit, bkg_val, 'k:', alpha=0.5, label="Local Bkg", zorder=2)
                
                # 各個別ピーク成分の描画
                for comp_name, comp_val in comps.items():
                    if "p" in comp_name:  # ピーク成分のみ抽出
                        label = f"{key.upper()} Comp ({comp_name.split('_')[0]})"
                        
                        # 【修正箇所】塗りつぶしの底をbkg_valとし、高さをbkg_val + comp_valにする
                        plt.fill_between(
                            x_fit, 
                            bkg_val, 
                            comp_val + bkg_val, 
                            alpha=0.2, 
                            label=label, 
                            zorder=2
                        )

        # --- 4. 補助情報の描画 ---
        # 範囲表示 (G, D, 2D)
        for rng, color, lbl in zip([g_range, d_range, two_d_range], 
                                   ["green", "blue", "red"], 
                                   ["G-range", "D-range", "2D-range"]):
            if rng is not None:
                plt.axvspan(*rng, color=color, alpha=0.05, label=lbl)

        # ピーク位置の数値的な強調
        if show_peak:
            def draw_max_peak(rng, label):
                if rng is None: return
                m = (wavenumber >= rng[0]) & (wavenumber <= rng[1])
                if m.any():
                    peak_wn = wavenumber[m][np.argmax(spectra[i, m])]
                    plt.axvline(peak_wn, color="red", linestyle=":", alpha=0.6)
                    plt.text(peak_wn, plt.ylim()[1]*0.9, f"{peak_wn:.1f}", 
                             color="red", rotation=90, verticalalignment='top')

            draw_max_peak(g_range, "G")
            draw_max_peak(d_range, "D")
            draw_max_peak(two_d_range, "2D")

        # レイアウト調整
        plt.title(f"Spectral Analysis - Pixel {i}")
        plt.xlabel(r"Wavenumber ($cm^{-1}$)")
        plt.ylabel("Intensity (a.u.)")
        plt.legend(loc='upper center', fontsize='x-small', ncol=2)
        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        
        # 描画範囲の制限
        plt.xlim(mask_range)
        
        # 保存とクローズ
        save_path = output_dir / f"{output_prefix}_pixel_{i}.png"
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()

In [12]:
def export_ratio_maps(
    filepath: Path,
    x,
    y,
    ig,
    id,
    i2d,
    ratio_gd,
    ratio_2dg,
):
    df_results = pd.DataFrame({
        'X': x, 'Y': y,
        'G_Intensity': ig,
        'D_Intensity': id,
        '2D_Intensity': i2d,
        'D_G_Ratio': ratio_gd,
        '2D_G_Ratio': ratio_2dg
    })
    df_results.to_csv(filepath, index=False)

In [13]:
def _fit_worker(i, wavenumber, spectrum):
    mask_g_search = (wavenumber >= 1550) & (wavenumber <= 1700)
    if np.any(mask_g_search):
        g_center_init = wavenumber[mask_g_search][np.argmax(spectrum[mask_g_search])]
    else:
        g_center_init = 1580

    dg_setup = [
        peaksetup(center=1350, weight=0.5, label="D"),
        peaksetup(center=g_center_init, weight=1.0, label="G")
    ]
    twod_setup = [peaksetup(center=2680, weight=1.0, label="2D")]

    try:
        mask_dg = (wavenumber >= D_PEAK_RANGE[0]) & (wavenumber <= G_PEAK_RANGE[1])
        res_dg = fit_single_spectrum(wavenumber[mask_dg], spectrum[mask_dg], dg_setup)
        
        mask_2d = (wavenumber >= TWO_D_PEAK_RANGE[0]) & (wavenumber <= TWO_D_PEAK_RANGE[1])
        res_2d = fit_single_spectrum(wavenumber[mask_2d], spectrum[mask_2d], twod_setup)
        
        val_d = res_dg.params['p0_height'].value
        val_g = res_dg.params['p1_height'].value
        val_2d = res_2d.params['p0_height'].value
        
        return i, val_d, val_g, val_2d, res_dg, res_2d
            
    except Exception as e:
        return i, 0, 0, 0, None, None


def calc_ratios_by_fitting(wavenumber, spectra):
    n_pixels = spectra.shape[0]
    I_D, I_G, I_2D = np.zeros(n_pixels), np.zeros(n_pixels), np.zeros(n_pixels)
    fit_results_dict = {}

    logger.info("Fitting all pixels using joblib multiprocessing...")

    results = Parallel(n_jobs=-1, verbose=5)(
        delayed(_fit_worker)(i, wavenumber, spectra[i])
        for i in range(n_pixels)
    )

    for i, val_d, val_g, val_2d, res_dg, res_2d in results: # type: ignore
        I_D[i] = val_d
        I_G[i] = val_g
        I_2D[i] = val_2d
        
        if res_dg is not None and res_2d is not None:
            fit_results_dict[i] = {"dg": res_dg, "2d": res_2d}
        elif val_d == 0 and val_g == 0 and val_2d == 0:
            pass 

    ratio_gd = I_D / (I_G + 1e-12)
    ratio_2dg = I_2D / (I_G + 1e-12)

    return ratio_2dg, ratio_gd, I_2D, I_G, I_D, fit_results_dict

In [14]:
def analyze_graphene_spectra(filepath: Path, d_g_ratio_min=0, d_g_ratio_max=None, two_d_g_ratio_min=0, two_d_g_ratio_max=None):
    save_dir = OUTPUT_DIR / filepath.relative_to(DATA_DIR).parent
    debug_path = save_dir / "debug"
    if debug_path.exists():
        for file in debug_path.glob("*.png"):
            file.unlink()

    x, y, wavenumber, spectra = load_mapping_ascii(filepath)
    
    available_idx = range(len(spectra))
    idx = np.random.choice(available_idx, min(5, len(available_idx)), replace=False)

    def run_stage(spectra, name, process_func, *args, **kwargs):
        spectra_before = spectra.copy()
        result = process_func(spectra, *args, **kwargs)
        
        if isinstance(result, tuple):
            spectra = result[0]
        else:
            spectra = result


        spectra_draw(
            wavenumber,
            spectra,
            spectra_ref=spectra_before,
            idx=idx,
            output_dir=debug_path,
            output_prefix=name,
            fit_results_dict=None,
        )
        return result

    spectra, shifts = run_stage(spectra, "wavenumber_corrected", correct_wavenumber_shift, wavenumber)
    spectra = run_stage(spectra, "baseline", preprocess_baseline, wavenumber, target_range=(1100, 3000))

    # -- 比計算 ---
    ratio_2dg, ratio_gd, I2D, IG, ID, fit_results_dict = calc_ratios_by_fitting(wavenumber, spectra)

    # --- マスク処理 ---
    mask = create_ig_mask(IG, method="absolute", threshold=5)
    ratio_2dg = apply_mask(ratio_2dg, mask)
    ratio_gd = apply_mask(ratio_gd, mask)

    # --- 結果の保存 ---
    export_ratio_maps(save_dir / "quantitative_results.csv", x, y, IG, ID, I2D, ratio_gd, ratio_2dg)

    # --- 2D/G比マップの保存 ---
    save_ratio_map_figure(
        save_dir / "2d_g_ratio_map.png",
        x,
        y,
        ratio_2dg,
        title="2D/G Ratio Map",
        vmin=np.nanpercentile(ratio_2dg, 25) if two_d_g_ratio_min is None else two_d_g_ratio_min,
        vmax=np.nanpercentile(ratio_2dg, 75) if two_d_g_ratio_max is None else two_d_g_ratio_max,
    )

    # --- D/G比マップの保存 ---
    save_ratio_map_figure(
        save_dir / "d_g_ratio_map.png",
        x,
        y,
        ratio_gd,
        title="D/G Ratio Map",
        vmin=np.nanpercentile(ratio_gd, 25) if d_g_ratio_min is None else d_g_ratio_min,
        vmax=np.nanpercentile(ratio_gd, 75) if d_g_ratio_max is None else d_g_ratio_max,
    )

    # --- デバッグ用スペクトル描画 ---
    spectra_draw(
        wavenumber,
        spectra,
        idx=idx,
        fit_results_dict=fit_results_dict,
        mask_range=(1100, 3000),
        g_range=G_PEAK_RANGE,
        d_range=D_PEAK_RANGE,
        two_d_range=TWO_D_PEAK_RANGE,
        show_peak=True,
        output_dir=debug_path,
        output_prefix="final_fit",
    )
    
    # --- 結果のログ出力 ---
    logger.info(f"D/G Ratio - min: {np.nanmin(ratio_gd):.3f}, max: {np.nanmax(ratio_gd):.3f}, median: {np.nanmedian(ratio_gd):.3f}")
    logger.info(f"2D/G Ratio - min: {np.nanmin(ratio_2dg):.3f}, max: {np.nanmax(ratio_2dg):.3f}, median: {np.nanmedian(ratio_2dg):.3f}")
    logger.info(f"Finished processing {filepath.name}")

In [15]:
D_G_RATIO_MIN = 0
D_G_RATIO_MAX = 1
TWO_D_G_RATIO_MIN = 0
TWO_D_G_RATIO_MAX = 2

source_dirs = [
    # DATA_DIR / "spectra_mapping" / "S1" / "S1_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S2" / "S2_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S4" / "S4_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S5" / "S5_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S7" / "S7_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S8" / "S8_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S11" / "SID-11_sub-si&hbn_aneal-1h_growth-2h_grating-100_acq-5x3.txt",
    # DATA_DIR / "spectra_mapping" / "S11b" / "S11b_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S11c" / "S11c_hole-100_x50_acq1sx1_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S11d" / "S11d_hole-100_x100_acq5sx1_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S12" / "SID-12_sub-si_aneal-1h_growth-2h_grating-100_acq-5x3_scale-x50.txt",
    # DATA_DIR / "spectra_mapping" / "S13" / "SID-13_sub-si_grating-100_acq-5x3_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S14" / "SID-14_grating-100_acq5x1_D03_hole-100_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S15" / "SID-15_grating-100_acq5x3_D03_hole-100_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S19" / "S19_hole-100_x100_acq5sx3_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S20a" / "S20a_hole-100_x100_acq5sx3_grating-100_D03.txt",
    DATA_DIR / "spectra_mapping" / "S24b" / "S24b_hole-100_x100_acq5sx3_grating-100_D03.txt",
]

for src in source_dirs:
    analyze_graphene_spectra(
        src,
        d_g_ratio_min=D_G_RATIO_MIN, 
        d_g_ratio_max=D_G_RATIO_MAX, 
        two_d_g_ratio_min=TWO_D_G_RATIO_MIN, 
        two_d_g_ratio_max=TWO_D_G_RATIO_MAX
    )

2026-05-18 13:37:40,202 - __main__ - INFO - Fitting all pixels using joblib multiprocessing...
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    1.7s
[Parallel(n_jobs=-1)]: Done 234 out of 234 | elapsed:    3.1s finished
2026-05-18 13:37:44,007 - __main__ - INFO - D/G Ratio - min: 0.040, max: 11.497, median: 0.110
2026-05-18 13:37:44,008 - __main__ - INFO - 2D/G Ratio - min: 0.294, max: 4.360, median: 1.093
2026-05-18 13:37:44,008 - __main__ - INFO - Finished processing S24b_hole-100_x100_acq5sx3_grating-100_D03.txt


In [16]:
D_G_RATIO_MIN = 0
D_G_RATIO_MAX = 1
TWO_D_G_RATIO_MIN = 0
TWO_D_G_RATIO_MAX = 2

source_dirs = [
    # DATA_DIR / "spectra_mapping" / "S1" / "S1_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S2" / "S2_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S4" / "S4_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S5" / "S5_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S7" / "S7_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S8" / "S8_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S11" / "SID-11_sub-si&hbn_aneal-1h_growth-2h_grating-100_acq-5x3.txt",
    # DATA_DIR / "spectra_mapping" / "S11b" / "S11b_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S11c" / "S11c_hole-100_x50_acq1sx1_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S11d" / "S11d_hole-100_x100_acq5sx1_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S12" / "SID-12_sub-si_aneal-1h_growth-2h_grating-100_acq-5x3_scale-x50.txt",
    # DATA_DIR / "spectra_mapping" / "S13" / "SID-13_sub-si_grating-100_acq-5x3_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S14" / "SID-14_grating-100_acq5x1_D03_hole-100_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S15" / "SID-15_grating-100_acq5x3_D03_hole-100_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S16" / "S16_sub-si_D03_acq5sx1_grating-100_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S17" / "S17_sub-si_D03_acq5sx1_grating-100_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S18" / "S18_sub-si_D03_acq5sx1_grating-100_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S19" / "S19_hole-100_x100_acq5sx3_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S20a" / "S20a_hole-100_x100_acq5sx3_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S23" / "S23_sub-si_D03_acq5sx3_grating-100_x100.txt",
    DATA_DIR / "spectra_mapping" / "S23" / "S23_D03_hole-100_grating-100_x100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "S24b" / "S24b_hole-100_x100_acq5sx3_grating-100_D03.txt",
]

for src in source_dirs:
    analyze_graphene_spectra(
        src,
        d_g_ratio_min=D_G_RATIO_MIN, 
        d_g_ratio_max=D_G_RATIO_MAX, 
        two_d_g_ratio_min=TWO_D_G_RATIO_MIN, 
        two_d_g_ratio_max=TWO_D_G_RATIO_MAX
    )

2026-05-18 13:37:45,740 - __main__ - INFO - Fitting all pixels using joblib multiprocessing...
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:    2.4s
[Parallel(n_jobs=-1)]: Done 270 out of 285 | elapsed:    3.4s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done 285 out of 285 | elapsed:    3.9s finished
2026-05-18 13:37:50,548 - __main__ - INFO - D/G Ratio - min: 0.000, max: 2.211, median: 0.145
2026-05-18 13:37:50,548 - __main__ - INFO - 2D/G Ratio - min: 0.082, max: 2.603, median: 1.487
2026-05-18 13:37:50,549 - __main__ - INFO - Finished processing S23_D03_hole-100_grating-100_x100_acq5sx3.txt


In [17]:
def plot_raman_data(x, y, wavenumber, spectra, peak, range):
    range_min = peak - range/2
    range_max = peak + range/2
    # ==========================================
    # 1. 平均スペクトルのプロット
    # ==========================================
    fig, ax = plt.subplots(figsize=(8, 5))
    mean_spectrum = np.mean(spectra, axis=0)

    ax.plot(wavenumber, mean_spectrum, color='b', linewidth=1)
    ax.axvspan(range_min, range_max, color='gray', alpha=0.2, label='Target Range')
    ax.set_xlabel("Raman Shift (cm⁻¹)", fontsize=16)
    ax.set_ylabel("Intensity (a.u.)", fontsize=16)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=16)
    plt.tight_layout()
    plt.show()

    # ==========================================
    # 2. 指定波数範囲のピーク強度マッピングのプロット
    # ==========================================
    mask = (wavenumber >= range_min) & (wavenumber <= range_max)
    if not np.any(mask):
        print(f"Warning: No data points found in range {range_min} - {range_max}")
        return
    peak_intensity = np.max(spectra[:, mask], axis=1)
    
    # 2次元グリッド化
    df_map = pd.DataFrame({'X': y, 'Y': x, 'Intensity': peak_intensity})
    pivot_table = df_map.pivot(index='Y', columns='X', values='Intensity')
    
    x_unique = pivot_table.columns.values
    y_unique = pivot_table.index.values
    C = pivot_table.values
    
    fig, ax = plt.subplots(figsize=(6, 5)) 
    sc = ax.pcolormesh(x_unique, y_unique, C, shading="auto", cmap='viridis')
    plt.colorbar(sc, label='Peak Max Intensity (a.u.)', shrink=0.6, aspect=20, pad=0.05)
    
    scalebar = ScaleBar(1, "um", length_fraction=0.2, location="lower right", 
                        box_alpha=0, color="white", font_properties={'size': 12})
    ax.add_artist(scalebar)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')
    ax.invert_yaxis() 

    plt.tight_layout()
    plt.show() # 2つ目の図を表示

# SOURCE_DIR = DATA_DIR / "spectra_mapping" 
# filepath = SOURCE_DIR / "expD" / "WSe2_Raman_D03_1s_1accumulation_50X.txt"
# x, y, wavenumber, spectra = load_mapping_ascii(filepath)
# plot_raman_data(x, y, wavenumber, spectra, peak=250, range=10)